In [ ]:
from torch.utils.data import Dataset

max_dataset_size = 200000

class LCSTS(Dataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)

    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'rt', encoding='utf-8') as f:
            for idx, line in enumerate(f):
                if idx >= max_dataset_size:
                    break
                items = line.strip().split('!=!') # 清理文本数据中的空格、换行符等。并按 !=! 把字符串分割成一个列表
                assert len(items) == 2
                #{0: {'title': '...', 'content': '...'}, 1: {'title': '...', 'content': '...'}...}
                Data[idx] = {
                    'title': items[0],
                    'content': items[1]
                }
        return Data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

train_data = LCSTS('data/lcsts_tsv/data1.tsv')
valid_data = LCSTS('data/lcsts_tsv/data2.tsv')
test_data = LCSTS('data/lcsts_tsv/data3.tsv')

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "csebuetnlp/mT5_multilingual_XLSum"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSeq2SeqLM

max_input_length = 512
max_target_length = 64

device = 'cuda' if torch.cuda.is_available else 'cpu'

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
model = model.to(device)

def collote_fn(batch_samples):
    batch_inputs, batch_targets = [], []
    # 从 sample 字典中取出 'content' 键对应的值（输入文本），添加到 batch_inputs 列表中
    for sample in batch_samples:
        batch_inputs.append(sample['content'])
        batch_targets.append(sample['title'])
    # 将输入文本 batch_inputs 转换为编码器可以接受的格式。
    batch_data = tokenizer(
        batch_inputs,
        padding = True,
        max_length = max_input_length,
        truncation = True,
        return_tensors = 'pt'
    )

    with tokenizer.as_target_tokenizer(): # 上下文管理器，再处理多语言或seq2seq任务时，确保使用适用于目标端的词汇表和特殊的token规则来处理batch_targets
        labels = tokenizer(
            batch_targets,
            padding = True,
            max_length = max_target_length,
            truncation=True,
            return_tensors="pt"
        )["input_ids"] # 只提取"input_ids”作为labels
        
        # 创建解码器的输入序列'decoder_input_ids'，需要整体右移一位。如果 labels 是 [<start>, t1, t2, <eos>]
        # 那么 decoder_input_ids 应该是 [<decoder_start>, <start>, t1, t2]
        batch_data['decoder_input_ids'] = model.prepare_decoder_input_ids_from_labels(labels)
        # 对labels进行处理，忽略填充部分对损失计算的影响
        end_token_index = torch.where(labels == tokenizer.eos_token_id)[1]
        for idx, end_idx in enumerate(end_token_index):
            # 将第 idx 个序列中，从结束符之后的所有位置（这些都是填充位）的值替换为 -100
            # 在 PyTorch 的交叉熵损失函数（nn.CrossEntropyLoss）中，默认的 ignore_index 参数值就是 -100
            labels[idx][end_idx+1:] = -100
        batch_data['labels'] = labels
    return batch_data

train_dataloader = DataLoader(train_data, batch_size=4, shuffle=True, collate_fn=collote_fn)
valid_dataloader = DataLoader(valid_data, batch_size=4, shuffle=False, collate_fn=collote_fn)

#### 最终得到的batch data 
{

    "input_ids": tensor(...),
    
    "attention_mask": tensor(...),

    "decoder_input_ids": tensor(...),

    "labels": tensor(...)
    
}

In [ ]:
from tqdm.auto import tqdm

def train_loop(dataloader, model, optimizer, lr_scheduler, epoch, total_loss):
    progress_bar = tqdm(range(len(dataloader)))
    progress_bar.set_description(f'loss: {0:>7f}')
    finish_batch_num = (epoch-1) * len(dataloader)
    
    model.train()
    for batch, batch_data in enumerate(dataloader, start=1):
        batch_data = batch_data.to(device)
        outputs = model(**batch_data)
        # 使用 AutoModelForSeq2SeqLM 构造的模型已经封装好了对应的损失函数，
        # 并且计算出的损失会直接包含在模型的输出 outputs 中，可以直接通过 outputs.loss 获得
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        progress_bar.set_description(f'loss: {total_loss/(finish_batch_num + batch):>7f}')
        progress_bar.update(1)
    return total_loss

In [ ]:
from rouge import Rouge

generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"

rouge = Rouge()

scores = rouge.get_scores(
    hyps=[generated_summary], refs=[reference_summary]
)[0]
print(scores)

可以通过 rouge 库来方便地计算这些 ROUGE 值，例如 ROUGE-1 度量 uni-grams 的重合情况，ROUGE-2 度量 bi-grams 的重合情况，而 ROUGE-L 则通过在生成摘要和参考摘要中寻找最长公共子串来度量最长的单词匹配序列。

rouge库默认用空格分词

scores输出结构（分别代表精确率p，召回率r，F1值f）：

{

    'rouge-1': {'r': 1.0, 'p': 0.8571, 'f': 0.9230},

    'rouge-2': {'r': 0.8, 'p': 0.6667, 'f': 0.7272},

    'rouge-l': {'r': 1.0, 'p': 0.8571, 'f': 0.9230}
    
}

In [ ]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device} device')

model_checkpoint = "csebuetnlp/mT5_multilingual_XLSum"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
model = model.to(device)

article_text = """
受众在哪里，媒体就应该在哪里，媒体的体制、内容、技术就应该向哪里转变。
媒体融合关键是以人为本，即满足大众的信息需求，为受众提供更优质的服务。
这就要求媒体在融合发展的过程中，既注重技术创新，又注重用户体验。
"""

input_ids = tokenizer(
    article_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)
# 柱搜索解码
generated_tokens = model.generate(
    input_ids["input_ids"],
    attention_mask=input_ids["attention_mask"],
    max_length=32,
    no_repeat_ngram_size=2, # 生成过程中，不允许任何长度为 2 的连续 token 片段重复出现
    num_beams=4 # 在 beam search 中，每一步最多保留 $4$ 条候选序列。
)
summary = tokenizer.decode(
    generated_tokens[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)
print(summary)

AutoModelForSeq2SeqLM 模型对 Decoder 的解码过程也进行了封装，我们只需要调用模型的 generate() 函数就可以自动地逐个生成预测 token

这里用到的柱搜索解码Beam Search 会在每一步保留多个得分最高的候选序列，而不是只保留一个。
假设 beam size 是 $k$，那么在每一轮：

每条候选序列都会继续扩展出很多新 token
所有扩展结果一起打分
最后只保留得分最高的 $k$ 条

$n$-gram 就是长度为 $n$ 的连续 token 序列。
例如句子：
“我 喜欢 自然 语言 处理”
它的：


$1$-gram 是单个 token：

“我”
“喜欢”
“自然”
“语言”
“处理”



$2$-gram 是连续两个 token：

“我 喜欢”
“喜欢 自然”
“自然 语言”
“语言 处理”



$3$-gram 是连续三个 token：

“我 喜欢 自然”
“喜欢 自然 语言”
“自然 语言 处理”





In [ ]:
import numpy as np
from rouge import Rouge

rouge = Rouge()

def test_loop(dataloader, model):
    preds, labels = [], []
    
    model.eval()
    for batch_data in tqdm(dataloader):
        batch_data = batch_data.to(device)
        with torch.no_grad():
            generated_tokens = model.generate(
                batch_data["input_ids"],
                attention_mask=batch_data["attention_mask"],
                max_length=max_target_length,
                num_beams=4,
                no_repeat_ngram_size=2,
            ).cpu().numpy()

        # 防御式编程，如果返回的是 tuple，就取第一个元素作为真正的生成 token 序列
        if isinstance(generated_tokens, tuple):
            generated_tokens = generated_tokens[0]
        
        label_tokens = batch_data["labels"].cpu().numpy()

        # 把预测token解码成文本，skip_special_tokens为解码时跳过特殊token
        decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        # 如果某个位置是 -100，替换成 tokenizer.pad_token_id
        label_tokens = np.where(label_tokens != -100, label_tokens, tokenizer.pad_token_id)
        # 把真实标签 token 解码成文本
        decoded_labels = tokenizer.batch_decode(label_tokens, skip_special_tokens=True)
        # pred.strip()：去掉字符串首尾空白；' '.join(pred.strip())：适配ROUGE，字符间加入空格
        preds += [' '.join(pred.strip()) for pred in decoded_preds]
        labels += [' '.join(label.strip()) for label in decoded_labels]
    
    scores = rouge.get_scores(hyps=preds, refs=labels, avg=True) # 计算分数，并返回整个测试集的平均值
    result = {key: value['f'] * 100 for key, value in scores.items()} # 取 F1 值并转换成百分比
    result['avg'] = np.mean(list(result.values())) # 在已有的 ROUGE 指标基础上计算平均值
    print(f"Rouge1: {result['rouge-1']:>0.2f} Rouge2: {result['rouge-2']:>0.2f} RougeL: {result['rouge-l']:>0.2f}\n")
    return result

In [ ]:
from transformers import AdamW, get_scheduler

learning_rate = 2e-5
epoch_num = 10

optimizer = AdamW(model.parameters(), lr=learning_rate)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=epoch_num*len(train_dataloader),
)

total_loss = 0.
best_avg_rouge = 0.
for t in range(epoch_num):
    print(f"Epoch {t+1}/{epoch_num}\n-------------------------------")
    total_loss = train_loop(train_dataloader, model, optimizer, lr_scheduler, t+1, total_loss)
    valid_rouge = test_loop(valid_dataloader, model)
    print(valid_rouge)
    rouge_avg = valid_rouge['avg']
    if rouge_avg > best_avg_rouge:
        best_avg_rouge = rouge_avg
        print('saving new weights...\n')
        # 保存当下最优模型到指定文件名的文件中
        torch.save(model.state_dict(), f'epoch_{t+1}_valid_rouge_{rouge_avg:0.4f}_model_weights.bin')
print("Done!")